# অধ্যায় ৭: চ্যাট অ্যাপ্লিকেশন তৈরি
## OpenAI API দ্রুত শুরু

এই নোটবুকটি [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) থেকে অভিযোজিত যা নোটবুকগুলি অন্তর্ভুক্ত করে যা [Azure OpenAI](notebook-azure-openai.ipynb) সেবাগুলিতে অ্যাক্সেস করে।

Python OpenAI API Azure OpenAI মডেলগুলোর সাথেও কাজ করে, কিছু সংশোধনের সাথে। এখানে পার্থক্যগুলি সম্পর্কে আরও জানুন: [Python দিয়ে OpenAI এবং Azure OpenAI এন্ডপয়েন্টসের মধ্যে কীভাবে সোয়িচ করবেন](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# ওভারভিউ  
"লরজ ল্যাঙ্গুয়েজ মডেলগুলি এমন ফাংশন যা টেক্সট থেকে টেক্সটে ম্যাপ করে। একটি ইনপুট স্ট্রিং টেক্সট দেওয়া হলে, একটি লরজ ল্যাঙ্গুয়েজ মডেল পরবর্তী আসা টেক্সটটি voorspraak করার চেষ্টা করে"(1)। এই "কুইকস্টার্ট" নোটবুক ব্যবহারকারীদের উচ্চ-স্তরের এলএলএম ধারণা, এএমএল দিয়ে শুরু করার জন্য মূল প্যাকেজ প্রয়োজনীয়তা, প্রম্পট ডিজাইনের একটি নরম পরিচিতি, এবং বিভিন্ন ব্যবহারক্ষেত্রের কয়েকটি সংক্ষিপ্ত উদাহরণ পরিচয় করিয়ে দেবে। 


## বিষয়বস্তু সূচি  

[ওভারভিউ](#overview)  
[OpenAI সার্ভিস কীভাবে ব্যবহার করবেন](#how-to-use-openai-service)  
[1. আপনার OpenAI সার্ভিস তৈরি করা](#1.-creating-your-openai-service)  
[2. ইনস্টলেশন](#2.-installation)    
[3. শংসাপত্র](#3.-credentials)  

[ব্যবহার কেসসমূহ](#use-cases)    
[1. টেক্সট সারাংশ তৈরি](#1.-summarize-text)  
[2. টেক্সট শ্রেণীবিন্যাস](#2.-classify-text)  
[3. নতুন পণ্যের নাম তৈরি](#3.-generate-new-product-names)  
[4. একটি শ্রেণীবিন্যাসকারী উন্নত করা](#4.fine-tune-a-classifier)  

[তথ্যসূত্র](#references)


### আপনার প্রথম প্রম্পট তৈরি করুন  
এই সংক্ষিপ্ত অনুশীলনটি একটি সাধারণ কাজ "সারসংক্ষেপ" এর জন্য OpenAI মডেলে প্রম্পট জমা দেওয়ার একটি মৌলিক পরিচয় প্রদান করবে।  


**ধাপসমূহ**:  
1. আপনার পাইথন পরিবেশে OpenAI লাইব্রেরি ইনস্টল করুন  
2. স্ট্যান্ডার্ড হেল্পার লাইব্রেরিগুলি লোড করুন এবং আপনি যে OpenAI সেবা তৈরি করেছেন তার জন্য আপনার সাধারণ OpenAI সিকিউরিটি ক্রিডেনশিয়াল সেট করুন  
3. আপনার কাজের জন্য একটি মডেল নির্বাচন করুন  
4. মডেলের জন্য একটি সহজ প্রম্পট তৈরি করুন  
5. মডেল API তে আপনার অনুরোধ জমা দিন!  


### 1. OpenAI ইনস্টল করুন


In [ ]:
%pip install openai python-dotenv

### ২. সহায়ক লাইব্রেরি আমদানি করুন এবং পরিচয়পত্র ইনস্ট্যানশিয়েট করুন


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### ৩. সঠিক মডেল খোঁজা  
GPT-4o এবং GPT-4o mini এর মতো মডেলগুলো প্রাকৃতিক ভাষা বুঝতে এবং তৈরি করতে পারে, এবং বিভিন্ন কাজের জন্য উপযুক্ত বিভিন্ন ক্ষমতা ও গতিতে ওপেনএআই প্ল্যাটফর্মে উপলব্ধ।  


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## ৪. প্রম্পট ডিজাইন  

"বৃহৎ ভাষা মডেলগুলোর জাদু হলো বিশাল পরিমাণ টেক্সটের উপর এই পূর্বাভাস ত্রুটি কমানোর জন্য প্রশিক্ষিত হওয়ার মাধ্যমে, মডেলগুলি এই পূর্বাভাসগুলোর জন্য উপযোগী ধারণাগুলো শিখে ফেলে। উদাহরণস্বরূপ, তারা এই ধরনের ধারণাগুলো শিখে"(1):

* কীভাবে বানান করতে হয়
* কীভাবে ব্যাকরণ কাজ করে
* কীভাবে পরিভাষা করতে হয়
* কীভাবে প্রশ্নের উত্তর দিতে হয়
* কীভাবে একটি কথোপকথন চালাতে হয়
* কীভাবে বহু ভাষায় লেখা যায়
* কীভাবে কোড লিখতে হয়
* ইত্যাদি।

#### কীভাবে একটি বড় ভাষা মডেল নিয়ন্ত্রণ করবেন  
"বড় ভাষা মডেলের সব ইনপুটের মধ্যে, সবচেয়ে প্রভাবশালী হলো টেক্সট প্রম্পট(1).

বড় ভাষা মডেলগুলোকে আউটপুট দিতে প্রম্পট করা যায় কয়েকটি উপায়ে:

নির্দেশনা: মডেলকে বলুন আপনি কী চান
সম্পূরক: মডেলকে আপনার চাওয়ার শুরুটা সম্পূর্ণ করতে উৎসাহিত করুন
প্রদর্শন: মডেলকে দেখান আপনি কী চান, যা হতে পারে:
প্রম্পটে কয়েকটি উদাহরণ
একটি ফাইন-টিউনিং প্রশিক্ষণ ডেটাসেটে শত বা হাজার হাজার উদাহরণ"



#### প্রম্পট তৈরির জন্য তিনটি মৌলিক নির্দেশিকা রয়েছে:

**দেখান এবং বলুন**। স্পষ্ট করুন আপনি কী চান, সেটা হতে পারে নির্দেশনা, উদাহরণ, বা দুটির সংমিশ্রণ দিয়ে। যদি আপনি চান মডেল একটি আইটেম তালিকাকে আলফাবেটিকাল অর্ডারে সাজাক বা একটি প্যারাগ্রাফকে আবেগ অনুসারে শ্রেণীবদ্ধ করুক, তবে দেখিয়ে দিন সেটাই আপনি চান।

**গুণমানসম্পন্ন ডেটা দিন**। আপনি যদি একটি শ্রেণীবিন্যাসকারী তৈরি করতে চেষ্টা করছেন বা মডেলকে একটি প্যাটার্ন অনুসরণ করাতে চান, নিশ্চিত করুন যে যথেষ্ট উদাহরণ আছে। আপনার উদাহরণগুলি প্রুফরিড করুন — মডেল সাধারণত বেসিক বানান ভুল দেখতে সক্ষম এবং আপনাকে একটি প্রতিক্রিয়া দেয়, কিন্তু এটি ভাবতে পারে যে এই ভুল ইচ্ছাকৃত, যা প্রতিক্রিয়াকে প্রভাবিত করতে পারে।

**আপনার সেটিংস চেক করুন।** তাপমাত্রা এবং top_p সেটিংস নিয়ন্ত্রণ করে মডেল কতটা নির্ধারিতভাবে প্রতিক্রিয়া তৈরি করছে। আপনি যদি এমন একটি প্রতিক্রিয়া চান যেখানে একটি সঠিক উত্তরই আছে, তবে এগুলো কম সেট করবেন। যদি আপনি বৈচিত্র্যময় প্রতিক্রিয়া চান, তবে এগুলো বেশি সেট করতে পারেন। সবচেয়ে বড় ভুল হলো এই সেটিংসগুলোকে "বুদ্ধিমত্তা" বা "সৃজনশীলতা" নিয়ন্ত্রণ মনে করা।


উৎস: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### ৫. জমা দিন!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### একই কলটি পুনরায় করুন, ফলাফলগুলি কেমন তুলনা হয়?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## টেক্সট সারাংশ  
#### চ্যালেঞ্জ  
একটি টেক্সট প্যাসেজের শেষে 'tl;dr:' যোগ করে টেক্সট সারাংশ তৈরি করুন। লক্ষ্য করুন কীভাবে মডেল অতিরিক্ত নির্দেশনা ছাড়াই বিভিন্ন কাজ করতে পারে। আপনি মডেলের আচরণ পরিবর্তন করতে এবং পেয়ে থাকা সারাংশকে কাস্টমাইজ করতে tl;dr থেকে আরও বর্ণনামূলক প্রম্পট দিয়ে পরীক্ষা করতে পারেন(3)।  

সাম্প্রতিক গবেষণায় প্রমাণিত হয়েছে যে বড় পরিমাণের টেক্সটের উপর প্রাক-প্রশিক্ষণ এবং পরে নির্দিষ্ট কাজের জন্য ফাইন-টিউনিং করা হলে অনেক NLP কাজ এবং বেঞ্চমার্কে উল্লেখযোগ্য উন্নতি হয়। যদিও সাধারণত স্থাপত্যে কাজ-অজ্ঞাত, এই পদ্ধতিটি এখনও হাজার হাজার বা দশ হাজারের অধিক উদাহরণের কাজ-নির্দিষ্ট ফাইন-টিউনিং ডেটাসেট প্রয়োজন। তুলনামূলকভাবে, মানুষেরা সাধারণত কয়েকটি উদাহরণ বা সহজ নির্দেশনা থেকে একটি নতুন ভাষার কাজ করতে পারে - যা বর্তমান NLP সিস্টেমগুলি এখনও বেশিরভাগ সময় করতে অক্ষম। এখানে আমরা দেখাই যে ভাষা মডেল বড় করার ফলে কাজ-অজ্ঞাত, কিছু-শট পারফরমেন্স ব্যাপকভাবে উন্নত হয়, কখনও কখনও পূর্ববর্তী সর্বোত্তম ফাইন-টিউনিং পদ্ধতির সাথে প্রতিযোগিতামূলক স্তরেও পৌঁছায়।  



Tl;dr  


# বিভিন্ন ব্যবহার ক্ষেত্রে অনুশীলন  
1. টেক্সট সারসংক্ষেপ করা  
2. টেক্সট শ্রেণীবদ্ধ করা  
3. নতুন পণ্যের নাম তৈরি করা


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## টেক্সট শ্রেণীবদ্ধ করুন  
#### চ্যালেঞ্জ  
আইটেমগুলোকে ইনফারেন্সের সময় প্রদত্ত শ্রেণীতে শ্রেণীবদ্ধ করুন। নিচের উদাহরণে, আমরা শ্রেণী ও শ্রেণীবদ্ধ করার জন্য টেক্সট উভয়ই প্রম্পটে প্রদান করি (*playground_reference)। 

গ্রাহকের অনুসন্ধান: হ্যালো, আমার ল্যাপটপের কীবোর্ডের একটি কী সম্প্রতি ভেঙে গেছে এবং আমাকে একটি পরিবর্তনের প্রয়োজন:

শ্রেণীবদ্ধকৃত বিভাগ:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## নতুন পণ্যের নাম তৈরি করুন
#### চ্যালেঞ্জ
উদাহরণ শব্দ থেকে পণ্যের নাম তৈরি করুন। এখানে আমরা এমন পণ্যের তথ্য পেশ করছি যাতে আমরা নাম তৈরি করতে যাচ্ছি। আমরা একটি অনুরূপ উদাহরণও দিচ্ছি যাতে প্যাটার্নটি বোঝা যায় যা আমরা পেতে চাই। আমরা র‍্যান্ডমনেস বাড়ানোর জন্য এবং আরও উদ্ভাবনী প্রতিক্রিয়া পেতে তাপমাত্রার মানও উচ্চ করে রেখেছি।

পণ্যের বিবরণ: একটি হোম মিল্কশেক মেকার
বীজ শব্দ: দ্রুত, স্বাস্থ্যকর, কমপ্যাক্ট।
পণ্যের নাম: HomeShaker, Fit Shaker, QuickShake, Shake Maker

পণ্যের বিবরণ: এমন একটি জুতো যার সাইজ যেকোনো পায়ের সাথে মানায়।
বীজ শব্দ: অভিযোজিত, মানানো, পুরো-ফিট।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# রেফারেন্স  
- [Openai কুকবুক](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry পোর্টাল](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [টেক্সট শ্রেণীবিন্যাসের জন্য GPT মডেল ফাইন-টিউনিংয়ের সেরা অনুশীলনসমূহ](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# আরও সাহায্যের জন্য  
[OpenAI কমার্শিয়ালাইজেশন টিম](AzureOpenAITeam@microsoft.com) 


# অবদানকারীরা
* লুইস লি  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
